# 02 · Possession chains
Inferred possessions, start/end types, and a hand-audit sample.

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import polars as pl, numpy as np
import matplotlib.pyplot as plt
from src.config import load_config
cfg = load_config()
proc = cfg.path(cfg.data.processed_dir)
pl.Config.set_tbl_rows(25); pl.Config.set_tbl_width_chars(160)

In [ ]:
poss = pl.read_parquet(proc / 'possessions.parquet')
print(poss.height, 'possessions;', round(poss.height / poss['game_id'].n_unique(), 1), 'per game')
poss.group_by('start_type').len().sort('len', descending=True)

In [ ]:
poss.group_by('end_type').len().sort('len', descending=True).head(15)

In [ ]:
poss.with_columns((pl.col('end_t') - pl.col('start_t')).alias('dur')).group_by('start_type').agg(pl.col('dur').median().alias('median_s'), pl.col('dur').quantile(0.9).alias('p90_s')).sort('start_type')

In [ ]:
obs = pl.read_parquet(proc / 'observations.parquet')
ev = pl.read_parquet(proc / 'events.parquet')
for r in poss.sample(3, seed=11).iter_rows(named=True):
    print('===', r['start_type'], '->', r['end_type'])
    print(obs.filter(pl.col('poss_id') == r['poss_id']).select('seq','obs_kind','player_id','t','x','y','strength'))
    print(ev.filter((pl.col('game_id') == r['game_id']) & pl.col('t_game').is_between(r['start_t'] - 2, r['end_t'] + 1)).select('t_game','type','team_id','player1','x','y'))